# Evaluación de los modelos (sin reentrenar)

**Proyecto final · Deep Learning**

Este notebook produce los análisis del Anteproyecto que **solo necesitan los checkpoints ya
entrenados** (`best.pt` y artefactos), sin volver a entrenar:

1. Carga de los 4 modelos desde `checkpoints/` y evaluación en el test de AG News.
2. **Matriz de confusión** y ejemplos mal clasificados por modelo.
3. **Protocolo de latencia** (batch 1 y 32, warmup + mediana).
4. **Conjunto externo (BBC News)**: mapeo de clases y **brecha de generalización**.
5. **Significancia estadística** entre modelos (McNemar).
6. **Distribución de la compuerta** de NewsHybridNet (in- vs. out-of-distribution).

> Requiere que exista la carpeta `checkpoints/` (generada por `train_models_kaggle.ipynb`).
> Si falta algún checkpoint, la celda correspondiente lo avisa y se salta ese modelo, en vez de
> romperse. Así el notebook es revisable antes de que termine el entrenamiento.

**Importante (NewsHybridNet):** fija `PYTHONHASHSEED` al mismo valor con el que se entrenó, porque la
rama léxica usa `hash()` para los n-gramas de caracteres.

## 0. Configuración, rutas y utilidades compartidas

Se reusan las mismas definiciones de preprocesamiento y arquitectura del notebook de entrenamiento.

In [ ]:
import os, json, re, time, math
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

# Rutas: busca checkpoints/ junto a este notebook (models/checkpoints) o en /kaggle.
CANDIDATES = [Path("checkpoints"), Path("models/checkpoints"), Path("/kaggle/working/checkpoints")]
CKPT_ROOT = next((p for p in CANDIDATES if p.exists()), CANDIDATES[0])
print("CKPT_ROOT:", CKPT_ROOT.resolve(), "| existe:", CKPT_ROOT.exists())

CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
NUM_CLASSES = len(CLASS_NAMES)

def ckpt_ready(name, files):
    d = CKPT_ROOT / name
    missing = [f for f in files if not (d / f).exists()]
    if missing:
        print(f"[SALTAR] {name}: faltan {missing}. Ejecuta primero el entrenamiento.")
        return False
    return True


In [ ]:
# --- Preprocesamiento (idéntico al notebook de entrenamiento) ---
PAD, UNK = "<pad>", "<unk>"
_token_re = re.compile(r"[a-z0-9]+|[^\sa-z0-9]")

def tokenize(text):
    return _token_re.findall(text.lower())

def encode(text, stoi, max_len, pad_idx):
    unk = stoi.get(UNK, 1)
    ids = [stoi.get(tok, unk) for tok in tokenize(text)][:max_len]
    length = len(ids)
    if length < max_len:
        ids = ids + [pad_idx] * (max_len - length)
    return ids, max(length, 1)

_norm_re = re.compile(r"\s+")
def _normalize_for_ngrams(text):
    return _norm_re.sub(" ", text.lower()).strip()

def char_ngram_hashes(text, buckets, ns):
    s = "<" + _normalize_for_ngrams(text) + ">"
    idxs = []
    for n in ns:
        if len(s) < n:
            continue
        for i in range(len(s) - n + 1):
            idxs.append(hash(s[i:i + n]) % buckets)
    return idxs or [0]


In [ ]:
# --- Definiciones de modelo (idénticas al notebook de entrenamiento) ---
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.context = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, h, mask):
        scores = self.context(torch.tanh(self.proj(h))).squeeze(-1)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(scores, dim=1)
        return torch.bmm(w.unsqueeze(1), h).squeeze(1), w

class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=200, hidden_dim=256, num_classes=NUM_CLASSES, pad_idx=0, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, 1, batch_first=True, bidirectional=True)
        self.attn = AdditiveAttention(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(nn.Linear(hidden_dim*2,128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128,num_classes))
        self.pad_idx = pad_idx
    def forward(self, x, lengths):
        mask = (x != self.pad_idx).long()
        emb = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        pooled, _ = self.attn(out, mask)
        return self.fc(self.dropout(pooled))

class SoftmaxRegression(nn.Module):
    def __init__(self, input_dim, num_classes=NUM_CLASSES):
        super().__init__(); self.linear = nn.Linear(input_dim, num_classes)
    def forward(self, x): return self.linear(x)

class SpatialDropout1d(nn.Module):
    def __init__(self, p): super().__init__(); self.drop = nn.Dropout2d(p)
    def forward(self, x):
        x = x.permute(0,2,1).unsqueeze(-1); x = self.drop(x); return x.squeeze(-1).permute(0,2,1)

class MaskedAdditiveAttention(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.proj = nn.Linear(dim, dim); self.context = nn.Linear(dim, 1, bias=False)
    def forward(self, h, mask):
        scores = self.context(torch.tanh(self.proj(h))).squeeze(-1)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(scores, dim=1)
        return torch.bmm(w.unsqueeze(1), h).squeeze(1), w

class NewsHybridNet(nn.Module):
    def __init__(self, emb_matrix, num_classes=NUM_CLASSES, pad_idx=0, ngram_buckets=2**17):
        super().__init__()
        vocab_size, emb_dim = emb_matrix.shape; self.pad_idx = pad_idx
        self.embedding = nn.Embedding.from_pretrained(emb_matrix, freeze=True, padding_idx=pad_idx)
        self.adapter = nn.Linear(emb_dim, 256); self.adapter_ln = nn.LayerNorm(256)
        self.spatial_drop = SpatialDropout1d(0.2)
        self.conv2 = nn.Conv1d(256,128,2,padding="same"); self.conv3 = nn.Conv1d(256,128,3,padding="same"); self.conv5 = nn.Conv1d(256,128,5,padding="same")
        self.cat_ln = nn.LayerNorm(384)
        self.bigru = nn.GRU(384,128,1,batch_first=True,bidirectional=True)
        self.attn = MaskedAdditiveAttention(256)
        self.embbag = nn.EmbeddingBag(ngram_buckets, 64, mode="mean")
        self.proj_a = nn.Linear(512,256); self.proj_b = nn.Linear(64,256); self.gate = nn.Linear(512,256)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(256,128), nn.GELU(), nn.Linear(128,num_classes))
    def forward(self, x, lengths, ngram_values, ngram_offsets, return_gate=False):
        mask = (x != self.pad_idx).long()
        emb = self.embedding(x)
        a = F.gelu(self.adapter_ln(self.adapter(emb))); a = self.spatial_drop(a)
        a_t = a.transpose(1,2)
        c = torch.cat([self.conv2(a_t), self.conv3(a_t), self.conv5(a_t)], dim=1).transpose(1,2)
        c = F.relu(self.cat_ln(c))
        packed = nn.utils.rnn.pack_padded_sequence(c, lengths.cpu(), batch_first=True, enforce_sorted=False)
        g_out, _ = self.bigru(packed)
        g_out, _ = nn.utils.rnn.pad_packed_sequence(g_out, batch_first=True, total_length=x.size(1))
        att, _ = self.attn(g_out, mask)
        masked = g_out.masked_fill((mask == 0).unsqueeze(-1), float("-inf"))
        maxp, _ = masked.max(dim=1); maxp = torch.nan_to_num(maxp, neginf=0.0)
        h_a = torch.cat([att, maxp], dim=1)
        h_b = self.embbag(ngram_values, ngram_offsets)
        ha_p = self.proj_a(h_a); hb_p = self.proj_b(h_b)
        g = torch.sigmoid(self.gate(torch.cat([ha_p, hb_p], dim=1)))
        fused = g * ha_p + (1 - g) * hb_p
        logits = self.head(fused)
        return (logits, g) if return_gate else logits


## 1. Datos de test de AG News y wrappers de predicción

Cada wrapper carga su `best.pt` y expone `predict_batch(textos) -> (preds, probs)`. Si falta el
checkpoint, el wrapper queda en `None` y los análisis lo omiten.

In [ ]:
from datasets import load_dataset

raw = load_dataset("fancyzhx/ag_news")
test_text = [t for t in raw["test"]["text"]]
test_label = [int(y) for y in raw["test"]["label"]]
print("test:", len(test_text))


In [ ]:
def _softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

class RnnWrapper:
    key, name = "rnn_scratch", "BiLSTM + atención"
    def __init__(self):
        d = CKPT_ROOT / self.key
        v = json.load(open(d / "vocab.json"))
        self.itos = v["itos"]; self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.max_len = v["max_len"]; self.pad_idx = v["pad_idx"]
        self.net = BiLSTMAttentionClassifier(len(self.itos), pad_idx=self.pad_idx).to(DEVICE)
        best = torch.load(d / "best.pt", map_location=DEVICE, weights_only=False)
        self.net.load_state_dict(best["model_state"]); self.net.eval()
    @torch.no_grad()
    def predict_batch(self, texts, bs=256):
        probs = []
        for i in range(0, len(texts), bs):
            chunk = texts[i:i+bs]
            enc = [encode(t, self.stoi, self.max_len, self.pad_idx) for t in chunk]
            x = torch.tensor([e[0] for e in enc], dtype=torch.long, device=DEVICE)
            ln = torch.tensor([e[1] for e in enc], dtype=torch.long, device=DEVICE)
            probs.append(_softmax_np(self.net(x, ln).cpu().numpy()))
        probs = np.concatenate(probs); return probs.argmax(1), probs

class TfidfWrapper:
    key, name = "tfidf_softmax", "TF-IDF + Softmax"
    def __init__(self):
        import joblib
        d = CKPT_ROOT / self.key
        self.vec = joblib.load(d / "tfidf_vectorizer.joblib")
        dim = len(self.vec.get_feature_names_out())
        self.net = SoftmaxRegression(dim).to(DEVICE)
        best = torch.load(d / "best.pt", map_location=DEVICE, weights_only=False)
        self.net.load_state_dict(best["model_state"]); self.net.eval()
    @torch.no_grad()
    def predict_batch(self, texts, bs=512):
        probs = []
        for i in range(0, len(texts), bs):
            X = self.vec.transform(texts[i:i+bs]).toarray().astype(np.float32)
            x = torch.from_numpy(X).to(DEVICE)
            probs.append(_softmax_np(self.net(x).cpu().numpy()))
        probs = np.concatenate(probs); return probs.argmax(1), probs

class HybridWrapper:
    key, name = "newshybridnet", "NewsHybridNet"
    def __init__(self):
        d = CKPT_ROOT / self.key
        v = json.load(open(d / "vocab.json"))
        self.itos = v["itos"]; self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.max_len = v["max_len"]; self.pad_idx = v["pad_idx"]
        ng = json.load(open(d / "ngram_config.json"))
        self.buckets = ng["buckets"]; self.ns = tuple(ng["ns"])
        emb = np.load(d / "embeddings.npy")
        self.net = NewsHybridNet(torch.tensor(emb), pad_idx=self.pad_idx, ngram_buckets=self.buckets).to(DEVICE)
        best = torch.load(d / "best.pt", map_location=DEVICE, weights_only=False)
        self.net.load_state_dict(best["model_state"]); self.net.eval()
    def _batch(self, chunk):
        enc = [encode(t, self.stoi, self.max_len, self.pad_idx) for t in chunk]
        x = torch.tensor([e[0] for e in enc], dtype=torch.long, device=DEVICE)
        ln = torch.tensor([e[1] for e in enc], dtype=torch.long, device=DEVICE)
        ng = [char_ngram_hashes(t, self.buckets, self.ns) for t in chunk]
        offs = torch.zeros(len(ng), dtype=torch.long, device=DEVICE)
        acc = 0
        for j,g in enumerate(ng): offs[j] = acc; acc += len(g)
        vals = torch.tensor([i for g in ng for i in g], dtype=torch.long, device=DEVICE)
        return x, ln, vals, offs
    @torch.no_grad()
    def predict_batch(self, texts, bs=256, return_gate=False):
        probs, gates = [], []
        for i in range(0, len(texts), bs):
            x, ln, vals, offs = self._batch(texts[i:i+bs])
            out = self.net(x, ln, vals, offs, return_gate=return_gate)
            logits, g = out if return_gate else (out, None)
            probs.append(_softmax_np(logits.cpu().numpy()))
            if return_gate: gates.append(g.mean(dim=1).cpu().numpy())
        probs = np.concatenate(probs); preds = probs.argmax(1)
        if return_gate: return preds, probs, np.concatenate(gates)
        return preds, probs

class BertWrapper:
    key, name = "distilbert", "DistilBERT"
    def __init__(self):
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        d = CKPT_ROOT / self.key
        self.tok = AutoTokenizer.from_pretrained(str(d / "tokenizer"))
        self.net = AutoModelForSequenceClassification.from_pretrained(str(d / "hf_best")).to(DEVICE)
        self.net.eval(); self.max_tokens = 128
    @torch.no_grad()
    def predict_batch(self, texts, bs=64):
        probs = []
        for i in range(0, len(texts), bs):
            enc = self.tok(texts[i:i+bs], truncation=True, max_length=self.max_tokens,
                           padding=True, return_tensors="pt").to(DEVICE)
            probs.append(_softmax_np(self.net(**enc).logits.cpu().numpy()))
        probs = np.concatenate(probs); return probs.argmax(1), probs

MODELS = {}
if ckpt_ready("tfidf_softmax", ["best.pt", "tfidf_vectorizer.joblib"]): MODELS["tfidf_softmax"] = TfidfWrapper()
if ckpt_ready("rnn_scratch", ["best.pt", "vocab.json"]):               MODELS["rnn_scratch"]   = RnnWrapper()
if ckpt_ready("newshybridnet", ["best.pt","vocab.json","embeddings.npy","ngram_config.json"]): MODELS["newshybridnet"] = HybridWrapper()
if ckpt_ready("distilbert", ["hf_best", "tokenizer"]):                 MODELS["distilbert"]    = BertWrapper()
print("Modelos cargados:", list(MODELS.keys()) or "ninguno (faltan checkpoints)")


In [ ]:
# Predicciones en el test de AG News (se cachean para los análisis siguientes).
from sklearn.metrics import accuracy_score, f1_score

y_true = np.array(test_label)
PRED = {}
for key, m in MODELS.items():
    preds, probs = m.predict_batch(test_text)
    PRED[key] = {"preds": preds, "probs": probs}
    print(f"{m.name:20s} acc={accuracy_score(y_true,preds):.4f}  macroF1={f1_score(y_true,preds,average='macro'):.4f}")
if not MODELS:
    print("Sin modelos: ejecuta el entrenamiento y coloca checkpoints/ junto a este notebook.")


## 2. Matriz de confusión y ejemplos mal clasificados

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    cmn = cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(5, 4.2))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real"); ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cmn[i,j]*100:.1f}", ha="center", va="center",
                    color="white" if cmn[i,j] > 0.5 else "black", fontsize=9)
    fig.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout(); plt.show()

for key, m in MODELS.items():
    print(f"\n===== {m.name} =====")
    yp = PRED[key]["preds"]
    print(classification_report(y_true, yp, target_names=CLASS_NAMES, digits=4))
    plot_confusion(y_true, yp, m.name)


In [ ]:
# Ejemplos mal clasificados (hasta 5 por modelo) con la clase real y la predicha.
for key, m in MODELS.items():
    yp = PRED[key]["preds"]
    wrong = np.where(yp != y_true)[0]
    print(f"\n===== {m.name}: {len(wrong)} errores de {len(y_true)} =====")
    for idx in wrong[:5]:
        print(f"  real={CLASS_NAMES[y_true[idx]]:8s} pred={CLASS_NAMES[yp[idx]]:8s} | {test_text[idx][:110]}")


## 3. Protocolo de latencia

Según la sección 6 del Anteproyecto: lote de tamaño 1 y 32, **50 ejecuciones de calentamiento
descartadas** y **mediana de las repeticiones**, incluyendo la tokenización. Se mide en el
`DEVICE` actual; para comparar CPU vs. GPU, reejecutar forzando `DEVICE`.

In [ ]:
import pandas as pd

def measure_latency(model, sample_texts, batch_size, warmup=50, reps=200):
    batch = (sample_texts * ((batch_size // len(sample_texts)) + 1))[:batch_size]
    for _ in range(warmup):
        model.predict_batch(batch)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        model.predict_batch(batch)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    med = float(np.median(times))
    return med, med / batch_size

sample = test_text[:32]
rows = []
for key, m in MODELS.items():
    for bs in (1, 32):
        total_ms, per_item = measure_latency(m, sample, bs, warmup=50, reps=200)
        rows.append({"modelo": m.name, "device": str(DEVICE), "batch": bs,
                     "mediana_ms_lote": round(total_ms, 3), "ms_por_noticia": round(per_item, 4)})
lat = pd.DataFrame(rows)
print(lat.to_string(index=False)) if rows else print("Sin modelos cargados.")
lat


## 4. Conjunto externo (BBC News) y brecha de generalización

Se carga BBC News, se mapean sus categorías a las de AG News (`business→Business`, `sport→Sports`,
`tech→Sci/Tech`, `politics→World` **condicional**, `entertainment` se excluye) y se mide la caída de
macro F1 respecto al test de AG News. Se reportan resultados **con y sin** la clase `politics`.

La celda intenta cargar BBC desde Hugging Face; si no hay internet, define `BBC_CSV_PATH` con un CSV
de columnas `text,category` y lo usa en su lugar.

In [ ]:
BBC_CSV_PATH = None  # p.ej. "bbc_news.csv" con columnas text,category
BBC_MAP = {"business": "Business", "sport": "Sports", "tech": "Sci/Tech", "politics": "World"}
LABEL2IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

def load_bbc():
    import pandas as pd
    if BBC_CSV_PATH and Path(BBC_CSV_PATH).exists():
        df = pd.read_csv(BBC_CSV_PATH)
        tcol = "text" if "text" in df.columns else df.columns[0]
        ccol = "category" if "category" in df.columns else df.columns[-1]
        return list(df[tcol].astype(str)), list(df[ccol].astype(str).str.lower())
    for name in [("SetFit/bbc-news", None), ("bbc_news", None)]:
        try:
            ds = load_dataset(name[0])
            split = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]
            tcol = "text" if "text" in split.column_names else split.column_names[0]
            ccol = "label_text" if "label_text" in split.column_names else ("category" if "category" in split.column_names else None)
            if ccol is None: continue
            return [str(t) for t in split[tcol]], [str(c).lower() for c in split[ccol]]
        except Exception as e:
            print(f"No se pudo cargar {name[0]}: {e}")
    return None, None

bbc_text_raw, bbc_cat_raw = load_bbc()
if bbc_text_raw is None:
    print("[SALTAR] No se pudo obtener BBC News. Define BBC_CSV_PATH con un CSV local.")
else:
    print("BBC cargado:", len(bbc_text_raw), "ejemplos | categorías:", sorted(set(bbc_cat_raw)))


In [ ]:
def build_bbc(include_politics):
    mapping = dict(BBC_MAP)
    if not include_politics: mapping.pop("politics", None)
    texts, labels = [], []
    for t, c in zip(bbc_text_raw, bbc_cat_raw):
        if c in mapping:
            texts.append(t); labels.append(LABEL2IDX[mapping[c]])
    return texts, np.array(labels)

if bbc_text_raw is not None and MODELS:
    ag_f1 = {key: f1_score(y_true, PRED[key]["preds"], average="macro") for key in MODELS}
    rows = []
    for include_politics in (True, False):
        bt, by = build_bbc(include_politics)
        tag = "con politics" if include_politics else "sin politics"
        for key, m in MODELS.items():
            preds, _ = m.predict_batch(bt)
            f1_bbc = f1_score(by, preds, average="macro")
            rows.append({"modelo": m.name, "escenario": tag, "n": len(bt),
                         "macroF1_AGNews": round(ag_f1[key], 4),
                         "macroF1_BBC": round(f1_bbc, 4),
                         "brecha": round(ag_f1[key] - f1_bbc, 4)})
    gen = pd.DataFrame(rows)
    print(gen.to_string(index=False))
else:
    print("Se omite: faltan BBC o modelos.")


## 5. Significancia estadística entre modelos (McNemar)

La prueba de McNemar compara dos clasificadores sobre los **mismos ejemplos** del test, usando los
casos en que uno acierta y el otro falla. Un p-valor bajo indica que la diferencia de aciertos no es
atribuible al azar.

In [ ]:
from itertools import combinations
from scipy.stats import chi2

def mcnemar(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b))   # A acierta, B falla
    c = int(np.sum(~correct_a & correct_b))   # A falla, B acierta
    if b + c == 0: return 0.0, 1.0, b, c
    stat = (abs(b - c) - 1) ** 2 / (b + c)    # con corrección de continuidad
    return stat, float(chi2.sf(stat, 1)), b, c

if len(MODELS) >= 2:
    correct = {key: (PRED[key]["preds"] == y_true) for key in MODELS}
    rows = []
    for ka, kb in combinations(MODELS, 2):
        stat, p, b, c = mcnemar(correct[ka], correct[kb])
        rows.append({"A": MODELS[ka].name, "B": MODELS[kb].name,
                     "A>B": b, "B>A": c, "chi2": round(stat, 3),
                     "p_value": f"{p:.2e}", "signif (p<0.05)": p < 0.05})
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("Se necesitan al menos 2 modelos para comparar.")


## 6. Distribución de la compuerta de NewsHybridNet (in- vs. out-of-distribution)

El Anteproyecto predice que, ante textos con muchas palabras fuera de vocabulario (conjunto externo),
la compuerta desplaza peso hacia la rama léxica (B). Se compara la distribución del valor medio de la
compuerta `g` (peso de la rama semántica A) en AG News (in-distribution) frente a BBC (out).

In [ ]:
if "newshybridnet" in MODELS:
    hyb = MODELS["newshybridnet"]
    _, _, gate_in = hyb.predict_batch(test_text[:3000], return_gate=True)
    print(f"AG News (in)  · g medio rama semántica: media={gate_in.mean():.3f}  std={gate_in.std():.3f}")
    plt.figure(figsize=(6, 3.6))
    plt.hist(gate_in, bins=30, alpha=0.6, label="AG News (in)", color="#6366f1")
    if bbc_text_raw is not None:
        bt, _ = build_bbc(include_politics=True)
        _, _, gate_out = hyb.predict_batch(bt, return_gate=True)
        print(f"BBC (out)     · g medio rama semántica: media={gate_out.mean():.3f}  std={gate_out.std():.3f}")
        plt.hist(gate_out, bins=30, alpha=0.6, label="BBC (out)", color="#22d3ee")
    plt.xlabel("g medio (peso rama semántica A)"); plt.ylabel("frecuencia")
    plt.title("Distribución de la compuerta de fusión"); plt.legend(); plt.tight_layout(); plt.show()
else:
    print("[SALTAR] NewsHybridNet no está cargado.")
